In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

from data_utils import load_CIFAR10
from linearsvm import linearSVM

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0)
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

ImportError: cannot import name 'linearSVM' from 'svm' (c:\Users\momo\Desktop\documentations\0329practice\svm.py)

In [ ]:
cifar10_dir = './cifar-10-data'
X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

print('训练集图片形状: ', X_train.shape)
print('训练集标签形状: ', y_train.shape)

classes =['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(classes)
samples_per_class = 7

for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

In [ ]:
#数据拆分
num_training = 49000
num_validation = 1000
num_test = 1000
num_dev = 500

X_val = X_train[num_training : num_training + num_validation]
y_val = y_train[num_training : num_training + num_validation]
X_train = X_train[:num_training]
y_train = y_train[:num_training]

X_dev = X_train[:num_dev]
y_dev = y_train[:num_dev]

X_test = X_test[:num_test]
y_test = y_test[:num_test]

X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_val = np.reshape(X_val, (X_val.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
X_dev = np.reshape(X_dev, (X_dev.shape[0], -1))

mean_image = np.mean(X_train, axis=0)
X_train -= mean_image
X_val -= mean_image
X_test -= mean_image
X_dev -= mean_image

X_train = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
X_val = np.hstack([X_val, np.ones((X_val.shape[0], 1))])
X_test = np.hstack([X_test, np.ones((X_test.shape[0], 1))])
X_dev = np.hstack([X_dev, np.ones((X_dev.shape[0], 1))])

print('处理完毕后，训练集矩阵的形状: ', X_train.shape)

In [ ]:
learning_rates=[1e-7,2e-7]
regularization_strengths=[2.5e4,5e4]
results={}
best_val=-1
best_svm=None

for lr in learning_rates:
    for reg in regularization_strengths:
        svm=linearSVM()
        loss_hist=svm.train(X_train,y_train,learning_rate=lr,reg=reg,num_iters=1500,verbose=False)
        y_train_pred=svm.predict(X_train)
        acc_train=np.mean(y_train==y_train_pred)
        y_val_pred = svm.predict(X_val)
        acc_val = np.mean(y_val == y_val_pred)

        results[(lr,reg)]=(acc_train,acc_val)
        print(f"lr: {lr}, reg: {reg} -> 验证集准确率: {acc_val:.4f}")

        if acc_val>best_val:
            best_val=acc_valbest_svm=svm

print(f'\n最佳验证集准确率 (Best Validation Accuracy): {best_val * 100:.2f}%')        

In [ ]:
y_test_pred = best_svm.predict(X_test)
test_accuracy = np.mean(y_test == y_test_pred)
print(f'期末考试准确率 (Test Accuracy): {test_accuracy * 100:.2f}%')

w = best_svm.W[:-1, :] 
w = w.reshape(32, 32, 3, 10) 
w_min, w_max = np.min(w), np.max(w)

classes =['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
plt.figure(figsize=(12, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)

    wimg = 255.0 * (w[:, :, :, i].squeeze() - w_min) / (w_max - w_min)
    plt.imshow(wimg.astype('uint8'))
    plt.axis('off')
    plt.title(classes[i])
plt.suptitle('Learned Weights Visualization (The "Ghost" Templates)', fontsize=16)
plt.show()